In [2]:
import serial
import time
import pandas as pd
import numpy as np

class ProstheticController:
    thumbClose = 25
    thumbOpen = 150
    indexClose = 18
    indexOpen = 180
    middleClose = 32
    middleOpen = 180
    ringClose = 152
    ringOpen = 0
    pinkyClose = 166
    pinkyOpen = 0
    def __init__(self, serial_port='COM5', baud_rate=115200):
        self.direction = {
            'thumb': 1,
            'index': 1,
            'middle': 1,
            'ring': 1,
            'pinky': 1
        }
        self.serial = serial.Serial(serial_port, baud_rate, timeout=1)
        time.sleep(2)
        self.scaling_coefficients = {
            'thumb': 1,
            'index': 1,
            'middle': 1,
            'ring': 1,
            'pinky': 1
        }
        self.servo_limits = (0, 160)
        self.channel_mapping = {
            1: 'thumb',
            2: 'index',
            3: 'middle',
            4: 'ring',
            5: 'pinky'
        }
        self.finger_to_idx = {
            'thumb': 0,
            'index': 1,
            'middle': 2,
            'ring': 3,
            'pinky': 4
        }
        
        # Define gestures and their corresponding servo positions
        self.gesture_poses = {
            'palm': {
                'thumb': self.thumbOpen,
                'index': self.indexOpen,
                'middle': self.middleOpen,
                'ring': self.ringOpen,
                'pinky': self.pinkyOpen
            },
            'index': {
                'thumb': self.thumbClose,
                'index': self.indexOpen,
                'middle': self.middleClose,
                'ring':  self.ringClose,
                'pinky': self.pinkyClose
            },
            'middle': {
                'thumb': self.thumbClose,
                'index': self.indexClose,
                'middle': self.middleOpen,
                'ring': self.ringClose,
                'pinky': self.pinkyClose
            },
            'pinky-ring': {
                'thumb': self.thumbClose,
                'index': self.indexClose,
                'middle': self.middleClose,
                'ring': self.ringOpen,
                'pinky': self.pinkyOpen
            },
            'fist': {
                'thumb': self.thumbClose,
                'index': self.indexClose,
                'middle': self.middleClose,
                'ring': self.ringClose,
                'pinky': self.pinkyClose
            },
            'thumb': {
                'thumb': self.thumbOpen,
                'index': self.indexClose,
                'middle': self.middleClose,
                'ring': self.ringClose,
                'pinky': self.pinkyClose
            }
        }

    def map_angle_to_servo(self, joint_angle_deg, finger_type):
        scaled = joint_angle_deg * self.scaling_coefficients[finger_type]
        clipped = np.clip(scaled, *self.servo_limits)

        if self.direction[finger_type] < 0:
            clipped = self.servo_limits[1] - (clipped - self.servo_limits[0])

        return clipped

    def send_to_arduino(self, angles_dict):
        parts = []
        for finger, angle in angles_dict.items():
            idx = self.finger_to_idx[finger]
            parts.append(f"SERVO:{idx}:{int(angle)}|")
        command_str = "".join(parts)
        self.serial.write(command_str.encode())
        print("Sent:", command_str)

    def process_gesture_timeline(self, timeline_df, start_offset=0.0, speedup=1.0):
        """Process gesture timeline CSV and execute defined poses"""
        print(f"Starting gesture timeline execution from {start_offset:.2f}s at {speedup:.1f}x speed...")
        
        # Sort and filter
        timeline_df = timeline_df.sort_values('start_time')
        timeline_df = timeline_df[timeline_df['start_time'] >= start_offset].reset_index(drop=True)
        
        start_ref_time = time.time()
        
        for i, row in timeline_df.iterrows():
            gesture = row['gesture']
            start_time = row['start_time'] - start_offset  # Re-zero timeline
            duration = row['duration'] / speedup  # Apply speedup

            current_time = time.time() - start_ref_time
            wait_time = start_time - current_time

            if wait_time > 0:
                print(f"Waiting {wait_time:.2f}s for gesture: {gesture}")
                time.sleep(wait_time)

            if gesture in self.gesture_poses:
                pose = self.gesture_poses[gesture]
                print(f"Executing {gesture} pose for {duration:.2f}s")
                self.send_to_arduino(pose)
                time.sleep(duration)
            else:
                print(f"Warning: No pose defined for gesture '{gesture}'")

        print("Gesture timeline execution completed")


    def close(self):
        """Cleanup serial connection"""
        self.serial.close()

if __name__ == "__main__":
    timeline_df = pd.read_csv("C:/Users/megab/Downloads/Gesture classification 4/PredictedTimelines/e.csv")
    
    # Ask user for start time and speedup factor
    try:
        start_offset = float(input("Enter start time offset in seconds (e.g. 400): ") or "0")
        speedup_factor = float(input("Enter speedup factor (e.g. 2.0 for 2x speed): ") or "1.0")
    except ValueError:
        print("Invalid input. Using default values.")
        start_offset = 0.0
        speedup_factor = 1.0

    arm = ProstheticController(serial_port="COM6", baud_rate=115200)
    
    try:
        arm.process_gesture_timeline(timeline_df, start_offset=start_offset, speedup=speedup_factor)
    finally:
        arm.close()


Starting gesture timeline execution from 0.00s at 1.0x speed...
Executing palm pose for 25.00s
Sent: SERVO:0:150|SERVO:1:180|SERVO:2:180|SERVO:3:0|SERVO:4:0|
Gesture timeline execution completed
